# BERTopic Setup — Protest Movement Indonesia (Aug–Sep 2025)
Research Workflow Phase 2: Thematic & Structural Modeling

**Jalankan setiap cell secara berurutan.**

Pastikan runtime sudah di-set ke GPU:
- Runtime → Change runtime type → GPU (T4)

## Cell 1: Instalasi Library
Jalankan cell ini sekali saja.

In [191]:
!pip install bertopic
!pip install sentence-transformers
!pip install umap-learn
!pip install hdbscan
!pip install Sastrawi

## Cell 2: Import Library

In [192]:
import pandas as pd
import numpy as np
from datetime import datetime

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

import warnings
warnings.filterwarnings("ignore")

print("Semua library berhasil di-import.")

Semua library berhasil di-import.


## Cell 3: Upload & Load Data

In [193]:
df = pd.read_excel("/content/Book6.xlsx")

# Parsing kolom date
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y %H:%M')

print(f"Total tweet: {len(df)}")
print(f"Periode: {df['date'].min()} — {df['date'].max()}")
print(f"Kolom: {list(df.columns)}")
print(f"\nDistribusi account_type:")
print(df['account_type'].value_counts())

Total tweet: 29289
Periode: 2025-08-01 00:01:00 — 2025-09-29 12:45:00
Kolom: ['date', 'is_quote', 'mentions', 'username', 'verified', 'account_type', 'full_text', 'tweet_url', 'quoted_url', 'view_count', 'quote_count', 'quoted_text', 'reply_count', 'display_name', 'retweet_count', 'favorite_count', 'in_reply_to_url', 'quoted_username', 'user_statuses_count', 'user_followers_count', 'user_following_count', 'in_reply_to_screen_name']

Distribusi account_type:
account_type
grassroot          28505
media/news           659
elite/political      125
Name: count, dtype: int64


## Cell 4: Persiapan Dokumen

In [194]:
# Drop baris yang full_text-nya kosong/NaN
df = df.dropna(subset=['full_text'])
df = df[df['full_text'].str.strip() != '']
df = df.reset_index(drop=True)

df['full_text'] = df['full_text'].astype(str)
docs = df['full_text'].tolist()
timestamps = df['date'].tolist()

print(f"Total dokumen setelah cleaning: {len(docs)}")
print(f"Contoh dokumen pertama:\n{docs[0]}")

Total dokumen setelah cleaning: 29289
Contoh dokumen pertama:
goodrecom ayo demo ke bankklau kita hanya diem aja ya mereka makin semena2


## Cell 5: Konfigurasi Stopwords Bahasa Indonesia

In [195]:
# Stopwords dari Sastrawi
factory = StopWordRemoverFactory()
sastrawi_stopwords = factory.get_stop_words()

# Tambahkan stopwords custom yang umum di Twitter Indonesia
custom_stopwords = [
    'yg','gak','ga','gw','lu','udah','udh','aja','nya','dong','sih','nih','deh','loh','lah','kan','kalo',
    'banget','bgt','emang','emg','gimana','gitu','gt','kyk','kayak','sm','sama','jg','juga','tp','tapi',
    'dr','dari','utk','untuk','dgn','dengan','krn','karena','blm','belum','sdh','sudah','tdk','tidak',
    'dlm','dalam','amp','rt','via',
    'a','aan','acab','acab acab','acabacab','ada','adalah','adanya','adapun','agar','agak','agaknya',
    'akhir','akhiri','akhirnya','aku','akulah','akan','akankah','amat','amatlah','an','anda','andalah',
    'antar','antara','antaranya','apa','apaan','apabila','apakah','apalagi','apatah','artinya','asal',
    'asalkan','atas','atau','ataukah','ataupun','awal','awalnya','bagai','bagaikan','bagaimana',
    'bagaimanakah','bagaimanapun','bagi','bagian','bahkan','bahwa','bahwasanya','baik','bakal','bakalan',
    'balik','banyak','bapak','baru','bawah','beberapa','begini','begitu','begitulah','begitupun','belakang',
    'belakangan','benar','berada','berakhir','berapa','berarti','berawal','berbagai','beri','berikan',
    'berikut','berikutnya','berjumlah','berkata','berlalu','berlangsung','berlebihan','bermacam',
    'bermaksud','bermula','bersama','bertanya','berturut','bertutur','berujar','berupa','besar','betul',
    'biasa','biasanya','bila','bisa','boleh','buat','bukan','bukannya','bulan','bung','cara','cukup',
    'cuma','d','dahulu','dan','dapat','daripada','datang','de','dekat','demi','demikian','depan','di',
    'dia','diantara','diberi','diberikan','dibuat','didapat','digunakan','diingat','diinginkan',
    'dikatakan','diketahui','dikira','dilakukan','dilalui','dilihat','dimaksud','diminta','dimulai',
    'dimungkinkan','dini','dipastikan','diperbuat','diperlukan','dipertanyakan','dipunyai','diri',
    'dirinya','disampaikan','disebut','disini','ditambahkan','ditanya','ditanyakan','ditegaskan',
    'ditujukan','ditunjuk','ditunjukkan','dituturkan','diucapkan','diungkapkan','dll','dst','dua','dulu',
    'empat','engga','enggak','entah','for','freepalestine','ftp','g','gini','gk','gm','gn','gue','guna',
    'gunakan','hal','hampir','hanya','hari','harus','hendak','hendaknya','hingga','ia','ialah','ibu',
    'ikut','ingat','ingin','ini','itu','jadi','jadinya','jam','jangan','jawab','jauh','jd','jelas',
    'jika','jumlah','justru','kah','kak','kala','kalau','kalian','kami','kamu','kan','kapan','karena',
    'kata','katanya','kau','ke','kecil','kedua','keinginan','kelamaan','kelihatan','keluar','kembali',
    'kemudian','kemungkinan','kenapa','kepada','keseluruhan','ketika','khususnya','kini','kira',
    'kita','kok','kurang','lagi','lagian','lain','lainnya','lalu','lama','lebih','lewat','lima','lo',
    'luar','macam','maka','makanya','makin','malah','mana','mampu','masa','masalah','masih','masing',
    'mau','maupun','melainkan','melakukan','melalui','melihat','memang','memberi','memberikan',
    'membuat','memerlukan','meminta','memperbuat','mempergunakan','memperkirakan','mempersiapkan',
    'mempertanyakan','mempunyai','memulai','memungkinkan','menambahkan','menanti','menanya',
    'menanyakan','mendapat','mendapatkan','mendatang','menegaskan','mengakhiri','mengapa',
    'mengatakan','mengenai','mengerjakan','mengetahui','menggunakan','mengingat','menginginkan',
    'mengira','mengucapkan','mengungkapkan','menjadi','menjawab','menjelaskan','menuju','menunjuk',
    'menunjukkan','menurut','menuturkan','menyampaikan','menyatakan','menyebutkan','menyeluruh',
    'menyiapkan','merasa','mereka','merupakan','meski','meskipun','meyakini','minta','misal',
    'misalnya','mula','mulai','mungkin','nah','naik','namun','nanti','nya','nyaris','nyatanya',
    'of','oleh','on','org','pada','padahal','pak','paling','para','pasti','pekan','penting','per',
    'percuma','perlu','pernah','persoalan','pertama','pertanyaan','pihak','place','post','pukul',
    'pula','pun','punya','rasa','rasanya','rata','rp','rupanya','s','saat','saja','saling','sambil',
    'sampai','sangat','satu','saya','se','sebab','sebagai','sebagaimana','sebagian','sebaiknya',
    'sebaliknya','sebelum','sebenarnya','seberapa','sebuah','secara','sedang','sedangkan','sedikit',
    'segala','segera','seharusnya','sehingga','sejak','sejumlah','sekadar','sekali','sekarang',
    'sekitar','selain','selalu','selama','selanjutnya','seluruh','semacam','semakin','sementara',
    'semisal','sempat','semua','sendiri','seolah','seorang','sepanjang','seperti','sering','serta',
    'sesaat','sesama','sesuatu','sesudah','setelah','setiap','setidaknya','seusai','si','siapa',
    'sini','soal','suatu','sudah','supaya','tadi','tahu','tahun','tak','tanpa','tanya','tapi',
    'tau','telah','tempat','tengah','tentang','tentu','tepat','terakhir','terasa','terdapat',
    'terdiri','terhadap','teringat','terjadi','terkira','terlalu','terlihat','termasuk','ternyata',
    'tersedia','tersebut','tertentu','terus','terutama','tetap','the','tiap','tiba','tidak','tiga',
    'tinggi','to','toh','tuh','turut','tutur','ucap','ujar','umum','ungkap','usah','usai',
    'video','waduh','wah','waktu','walau','walaupun','wong','ya','yaitu','yakin','yakni','yang','you',
    'pengen', 'bare', 'ni', 'all', 'dengerin', 'penuhi', 'kegblgnunfaedh', 'are', 'lg', 'yuk',
    'masi', 'gua', 'masak', 'ayo', 'dok', 'kt', 'ka', 'kak', 'go', 'ospek', 'biar', 'la', 'at',
    'member', 'dah', 'kudu', 'km', 'dimana2', 'mw', 'mau', 'saatnya', 'sekalian', 'kh', 'by', 'min',
    'kena', 'besarbesaran', 'mah', 'sek', 'mbak', 'mba', 'nantangin', 'umur', 'ehh', 'eh', 'ehhh', 'da', 'ah',
    'iki', 'wes', 'opo', 'sing', 'cantik', 'yok', 'ae', 'nek', 'lagu', 'ig', 'promo', 'receh', 'job',
    'saldo', 'diskon', 'done', 'yo', 'ra', 'nu', 'kabeh', 'zonauang', 'apk', 'do', 'one', 'piece', 'anime',
    'seribu', 'serba', 'rek', 'plus', '100', 'karo', 'akun', 'arhan', 'than', 'grahadi', 'spesial', 'kode',
    'model', 'masukin', 'gaza', 'pagi', 'setia', 'tikus', 'pas', 'seng', 'iku', 'kon', 'lek', 'in', 'is',
    'afif', 'bastards', 'as', 'now', 'me', 'story', 'ne', 'ha', 'ki', 'dadi', 'sg', 'sak', 'ben', 'aya',
    'na',

    # tambahan slang / sosial media
    'bikin','bkn','bnr','bener','bang','bro','cuy','guys','pls','plis','btw','jir','anjingg',
    'kek','kaya','kayanya','kayaknya','kemaren','beneran','sumpah','open','gas','langsung',
    'liat','ntar','sampe','mending','kasih','bawa','ikut','turun','doang','iya','eh','yah',

    # bahasa jawa
    'ora', 'opo', 'mbok', 'mboh', 'mosok', 'lan', 'karo', 'nang', 'ning', 'marga', 'kanggo', 'arep',
    'kudu', 'gelem', 'iso', 'ojo', 'wes', 'wis', 'isih', 'durung', 'mung', 'maneh', 'banget', 'tenan',
    'mas', 'mbak', 'mbah', 'eyang', 'piye', 'sopo', 'ngendi', 'ngono', 'ngene', 'aku', 'awakmu', 'kowe',
    'njenengan', 'kuwi', 'iku', 'iki', 'kae', 'to', 'ta', 'tho', 'yo', 'yok', 'lho', 'se', 'ae', 'wae',
    'bae', 'tok', 'men', 'kui', 'nek', 'onok', 'saiki'
]

indonesian_stopwords = list(set(sastrawi_stopwords + custom_stopwords))

print(f"Total stopwords: {len(indonesian_stopwords)}")

Total stopwords: 741


## Cell 6: Konfigurasi Komponen BERTopic
Pilih embedding model di sini. Default: `indo-sentence-bert-base`.

Untuk perbandingan, uncomment opsi lain dan jalankan ulang dari cell ini.

In [196]:
# --- Embedding Model ---
# Pilih salah satu, comment yang tidak dipakai

# Opsi 1: Multilingual (baseline)
# embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Opsi 2: Indonesian-specific sentence transformer
embedding_model = SentenceTransformer("firqaaa/indo-sentence-bert-base")

print(f"Embedding model loaded.")

# --- UMAP (Dimensionality Reduction) ---
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# --- HDBSCAN (Clustering) ---
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric='euclidean',
    prediction_data=True
)

# --- CountVectorizer (c-TF-IDF representation) ---
vectorizer_model = CountVectorizer(
    min_df=5,
    ngram_range=(1, 1),
    stop_words=indonesian_stopwords
)

print("Semua komponen BERTopic sudah dikonfigurasi.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: firqaaa/indo-sentence-bert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.
Semua komponen BERTopic sudah dikonfigurasi.


## Cell 7: Jalankan BERTopic
Proses ini memakan waktu 5–15 menit tergantung GPU Colab.

In [197]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

print(f"SEBELUM reduce outliers:")
print(f"Jumlah topik: {len(set(topics)) - 1}")
print(f"Outlier: {topics.count(-1)} ({topics.count(-1) / len(topics) * 100:.1f}%)")

# Reduce outliers menggunakan probabilitas
new_topics = topic_model.reduce_outliers(docs, topics, probabilities=probs, strategy="probabilities", threshold=0.05)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer_model)
topics = new_topics

print(f"\nSETELAH reduce outliers:")
print(f"Jumlah topik: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"Outlier tersisa: {topics.count(-1)} ({topics.count(-1) / len(topics) * 100:.1f}%)")

2026-04-21 09:14:21,236 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/916 [00:00<?, ?it/s]

2026-04-21 09:15:50,442 - BERTopic - Embedding - Completed ✓
2026-04-21 09:15:50,443 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-21 09:16:32,353 - BERTopic - Dimensionality - Completed ✓
2026-04-21 09:16:32,355 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-21 09:16:46,581 - BERTopic - Cluster - Completed ✓
2026-04-21 09:16:46,592 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-21 09:16:47,156 - BERTopic - Representation - Completed ✓
2026-04-21 09:16:47,670 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


SEBELUM reduce outliers:
Jumlah topik: 70
Outlier: 14722 (50.3%)

SETELAH reduce outliers:
Jumlah topik: 70
Outlier tersisa: 6751 (23.0%)


In [198]:
topic_model.reduce_topics(docs, nr_topics=20)
topics = topic_model.topics_
probs = topic_model.probabilities_

print(f"SETELAH merge:")
print(f"Jumlah topik: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"Outlier tersisa: {topics.count(-1)} ({topics.count(-1) / len(topics) * 100:.1f}%)")

2026-04-21 09:16:48,318 - BERTopic - Topic reduction - Reducing number of topics
2026-04-21 09:16:48,520 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-21 09:16:49,591 - BERTopic - Representation - Completed ✓
2026-04-21 09:16:49,602 - BERTopic - Topic reduction - Reduced number of topics from 71 to 20


SETELAH merge:
Jumlah topik: 19
Outlier tersisa: 6751 (23.0%)


## Cell 8: Lihat Hasil Topik

In [199]:
topic_info = topic_model.get_topic_info()
print(topic_info)

    Topic  Count                                               Name  \
0      -1   6751                        -1_demo_rakyat_dpr_tuntutan   
1       0   6095                        0_demo_dpr_gedung_mahasiswa   
2       1   2683   1_indonesiagelap_resetindonesia_indonesia_negara   
3       2   2506                    2_prabowo_dpr_presiden_tuntutan   
4       3   2124  3_polisi_polisipembunuh_polisipembunuhrakyat_1312   
5       4   2046                       4_rakyat_tuntutan_pajak_gaji   
6       5   1595                    5_affan_kurniawan_almarhum_ojol   
7       6   1180                          6_cops_1312_bastard_tolol   
8       7    807                  7_1312_pembunuh_forever_selamanya   
9       8    772  8_bubarkandprsontoloyo_berlakukanhukumanmati_b...   
10      9    543                    9_stay_safe_jaga_resetindonesia   
11     10    453                           10_temen_demo_1312_orang   
12     11    403                      11_anjing_polisi_1312_bangsat   
13    

## Cell 9: Detail Kata Representatif Per Topik

In [200]:
for topic_id in range(min(20, len(set(topics)) - 1)):
    print(f"\n--- Topic {topic_id} ---")
    topic_words = topic_model.get_topic(topic_id)
    for word, score in topic_words[:5]:
        print(f"  {word}: {score:.4f}")


--- Topic 0 ---
  demo: 0.0807
  dpr: 0.0484
  gedung: 0.0350
  mahasiswa: 0.0231
  ri: 0.0191

--- Topic 1 ---
  indonesiagelap: 0.0924
  resetindonesia: 0.0828
  indonesia: 0.0526
  negara: 0.0426
  prabowo: 0.0402

--- Topic 2 ---
  prabowo: 0.0606
  dpr: 0.0554
  presiden: 0.0546
  tuntutan: 0.0521
  rakyat: 0.0454

--- Topic 3 ---
  polisi: 0.1072
  polisipembunuh: 0.0860
  polisipembunuhrakyat: 0.0652
  1312: 0.0483
  bubarkandpr: 0.0387

--- Topic 4 ---
  rakyat: 0.0829
  tuntutan: 0.0743
  pajak: 0.0483
  gaji: 0.0440
  tunjangan: 0.0358

--- Topic 5 ---
  affan: 0.1649
  kurniawan: 0.1639
  almarhum: 0.0616
  ojol: 0.0616
  brimob: 0.0593

--- Topic 6 ---
  cops: 0.0781
  1312: 0.0738
  bastard: 0.0370
  tolol: 0.0334
  bajingan: 0.0317

--- Topic 7 ---
  1312: 0.5244
  pembunuh: 0.0343
  forever: 0.0336
  selamanya: 0.0312
  sakit: 0.0233

--- Topic 8 ---
  bubarkandprsontoloyo: 0.2590
  berlakukanhukumanmati: 0.2161
  bubarkandpr: 0.1850
  bubarkandprri: 0.0619
  ijazah: 0.

## Cell 10: Visualisasi — Intertopic Distance Map

In [201]:
fig_distance = topic_model.visualize_topics()
fig_distance.show()

## Cell 11: Visualisasi — Bar Chart Kata Per Topik

In [202]:
fig_barchart = topic_model.visualize_barchart(top_n_topics=20, n_words=5)
fig_barchart.show()

## Cell 12: Topics Over Time
Ini yang paling penting untuk riset — menunjukkan kapan topic shift terjadi.

In [203]:
topics_over_time = topic_model.topics_over_time(
    docs, # Changed from processed_docs to docs
    timestamps,
    nr_bins=20
)

fig_time = topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=8
)
fig_time.show()

20it [00:01, 17.42it/s]


## Cell 13: Simpan Hasil ke DataFrame

In [205]:
# Tambahkan kolom topic dan probability ke dataframe
df['topic'] = topics
# Since probs is a 1D array of max probabilities already:
df['probability'] = np.max(probs, axis=1)

# Tambahkan label topik
topic_labels = topic_model.generate_topic_labels(nr_words=3, separator=" | ")
topic_label_map = {i: label for i, label in enumerate(topic_labels)}
topic_label_map[-1] = "Outlier"
df['topic_label'] = df['topic'].map(topic_label_map)

print("Kolom baru ditambahkan: topic, probability, topic_label")
print(f"\nDistribusi topik:")
print(df['topic'].value_counts().head(10))

Kolom baru ditambahkan: topic, probability, topic_label

Distribusi topik:
topic
-1    6751
 0    6095
 1    2683
 2    2506
 3    2124
 4    2046
 5    1595
 6    1180
 7     807
 8     772
Name: count, dtype: int64


## Cell 14: Export Hasil

In [206]:
# Simpan dataframe dengan topic labels
df.to_csv("Data_bert_with_topics.csv", index=False)

# Download file
from google.colab import files
files.download("Data_bert_with_topics.csv")

print("File 'Data_bert_with_topics.csv' berhasil disimpan dan di-download.")
print("File ini akan digunakan untuk Langkah 2: Konstruksi Network.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'Data_bert_with_topics.csv' berhasil disimpan dan di-download.
File ini akan digunakan untuk Langkah 2: Konstruksi Network.


## Cell 15: Simpan Model (Opsional)
Untuk reproducibility — bisa di-load kembali tanpa harus fit ulang.

In [ ]:
topic_model.save("bertopic_model_protest")
print("Model BERTopic disimpan di folder 'bertopic_model_protest'")

# Untuk load kembali nanti:
# loaded_model = BERTopic.load("bertopic_model_protest")